# Grad-CAM Interpretability Analysis — CoAtNet IPP

Applies **Grad-CAM** (Selvaraju et al. 2017) to the best-performing IPP model,
**CoAtNet-0**, to verify that predictions rely on biologically meaningful
morphological cues rather than dataset-specific artefacts.

For each of the 10 representative spheroids, we generate one heatmap per
protocol attribute (8 total), arranged in a 2×4 grid (Fig. 5 in the paper).

**Sample coverage** spans diverse cell lines (A549, HCT116, PANC1, SKOV3,
U251MG, SW837, MCF10A, CT5.3hTERT), media, formation protocols (ULA, Hanging
Drop, Agarose, Microchip), seeding densities, timepoints, replicates, and
magnifications.

In [ ]:
# !pip install timm grad-cam --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import tifffile
import matplotlib.pyplot as plt
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import timm

In [ ]:
# Config
class CFG:
    metadata_csv = "../data/slimia_metadata.csv"
    ckpt_path    = "../checkpoints/ipp/CoAtNet-0_seed42.pth"
    encoder_dir  = "../results/ipp/"
    output_dir   = "../results/figures/gradcam/"

    label_columns = [
        "microscope", "cell_line", "culture_medium", "formation_method",
        "seeding_density", "timepoint", "biological_rep", "magnification"
    ]

    image_size = 224
    device     = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.output_dir, exist_ok=True)

val_tf = T.Compose([
    T.Resize((CFG.image_size, CFG.image_size)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),
])

## Load Model & Label Encoders

In [ ]:
# MultiTaskBackbone
class MultiTaskBackbone(nn.Module):
    def __init__(self, backbone_name, label_dims, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0)
        D = self.backbone.num_features
        self.heads = nn.ModuleDict({
            lab: nn.Linear(D, dim) for lab, dim in label_dims.items()
        })

    def forward(self, x):
        feat = self.backbone(x)
        return {lab: head(feat) for lab, head in self.heads.items()}


# Load label encoders & dims
label_encoders = {}
label_dims     = {}
for col in CFG.label_columns:
    le = joblib.load(os.path.join(CFG.encoder_dir, f"label_encoder_{col}.pkl"))
    label_encoders[col] = le
    label_dims[col]     = len(le.classes_)

print("Label dims:", label_dims)

# Load CoAtNet checkpoint
model = MultiTaskBackbone("coatnet_0_224", label_dims, pretrained=False)
model.load_state_dict(torch.load(CFG.ckpt_path, map_location=CFG.device))
model = model.to(CFG.device).eval()
print("Loaded CoAtNet-0 checkpoint.")

## Grad-CAM Implementation

CoAtNet's later stages use attention blocks that still output spatial feature
maps `(B, C, H, W)`, so standard Grad-CAM applies directly. We hook the output
of the **last stage** of the backbone (before global pooling).

In [ ]:
class GradCAM:
    """
    Minimal Grad-CAM implementation (no external dependency required).

    Hooks a target layer's forward activations and backward gradients,
    then computes:
        weights = global_avg_pool(grad)
        cam     = relu( sum_k weights_k * activation_k )
    """
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model        = model
        self.activations  = None
        self.gradients    = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x: torch.Tensor, label: str, class_idx: int = None):
        """
        Args:
            x: input tensor (1, 3, H, W)
            label: which protocol attribute head to explain
            class_idx: target class index; if None, uses the predicted class

        Returns:
            cam: (H, W) numpy array normalised to [0, 1]
            pred_idx: the predicted (or specified) class index
        """
        self.model.zero_grad()
        outputs = self.model(x)
        logits  = outputs[label]

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        score = logits[0, class_idx]
        score.backward(retain_graph=True)

        # Activations / gradients: (1, C, H, W) — handle both NCHW and NHWC
        acts  = self.activations
        grads = self.gradients
        if acts.shape[1] != grads.shape[1] and acts.shape[-1] == grads.shape[-1]:
            # NHWC → NCHW
            acts  = acts.permute(0, 3, 1, 2)
            grads = grads.permute(0, 3, 1, 2)

        weights = grads.mean(dim=(2, 3), keepdim=True)        # (1, C, 1, 1)
        cam     = F.relu((weights * acts).sum(dim=1))         # (1, H, W)
        cam     = cam.squeeze(0).cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


# Locate target layer
# timm's coatnet_0_224 exposes `.stages` — use the final stage's output.
def get_target_layer(model):
    backbone = model.backbone
    if hasattr(backbone, "stages"):
        return backbone.stages[-1]
    # Fallback: last module with parameters
    return list(backbone.children())[-1]

target_layer = get_target_layer(model)
gradcam = GradCAM(model, target_layer)
print(f"Grad-CAM hooked on: {type(target_layer).__name__}")

## Image Loading & Overlay Helpers

In [ ]:
def load_image_rgb(path: str) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        arr = tifffile.imread(path)
        if arr.ndim == 2:
            arr = np.stack([arr]*3, axis=-1)
        arr = arr.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        return Image.fromarray((arr*255).astype(np.uint8)).convert("RGB")


def overlay_cam(img_pil: Image.Image, cam: np.ndarray, alpha: float = 0.45):
    """Overlay a Grad-CAM heatmap on the original image."""
    img    = np.array(img_pil.resize((CFG.image_size, CFG.image_size))) / 255.0
    cam_r  = np.array(Image.fromarray((cam*255).astype(np.uint8))
                      .resize((CFG.image_size, CFG.image_size), Image.BILINEAR)) / 255.0
    heat   = plt.cm.jet(cam_r)[..., :3]
    blend  = (1 - alpha) * img + alpha * heat
    return np.clip(blend, 0, 1)

## Select 10 Representative Spheroids

In [ ]:
df = pd.read_csv(CFG.metadata_csv)
df["full_path"] = df["full_path"].astype(str).str.strip()

for col in CFG.label_columns:
    df[col] = df[col].astype(str)
    df[col + "_enc"] = label_encoders[col].transform(df[col])

# Target cell lines from the paper's interpretability sample
target_cell_lines = ["A549", "HCT116", "PANC1", "SKOV3",
                     "U251MG", "SW837", "MCF10A", "CT5.3hTERT"]

samples = []
for cl in target_cell_lines:
    sub = df[df["cell_line"] == cl]
    if len(sub) > 0:
        samples.append(sub.sample(1, random_state=42).iloc[0])

# Pad to 10 with random diverse samples (different formation methods)
while len(samples) < 10:
    extra = df.sample(1, random_state=len(samples)).iloc[0]
    samples.append(extra)

samples = samples[:10]
print(f"Selected {len(samples)} representative spheroids")
for s in samples:
    print(f"  {s['cell_line']:>14} | {s['culture_medium']:>10} | "
          f"{s['formation_method']:>12} | {s['timepoint']}")

## Generate Grad-CAM Grids (Fig. 5)

In [ ]:
DISPLAY_NAMES = {
    "cell_line":        "Cell Line",
    "culture_medium":   "Culture Medium",
    "formation_method": "Formation Method",
    "seeding_density":  "Seeding Density",
    "timepoint":        "Timepoint",
    "biological_rep":   "Biological Rep",
    "magnification":    "Magnification",
    "microscope":       "Microscope",
}

# Layout matches Fig 5: 2 rows x 4 cols
GRID_ORDER = [
    "cell_line", "culture_medium", "formation_method", "seeding_density",
    "timepoint", "biological_rep", "magnification", "microscope",
]
GRID_ORDER = [l for l in GRID_ORDER if l in CFG.label_columns]


def gradcam_grid_for_sample(row, save_name=None):
    img_pil = load_image_rgb(row["full_path"])
    x       = val_tf(img_pil).unsqueeze(0).to(CFG.device)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    for ax, label in zip(axes.flat, GRID_ORDER):
        cam, pred_idx = gradcam(x, label)
        overlay       = overlay_cam(img_pil, cam)
        ax.imshow(overlay)

        pred_class = label_encoders[label].inverse_transform([pred_idx])[0]
        true_class = row[label]
        correct    = "✓" if str(pred_class) == str(true_class) else "✗"

        ax.set_title(f"{DISPLAY_NAMES.get(label, label)}\n"
                     f"pred: {pred_class} {correct}", fontsize=9)
        ax.axis("off")

    plt.suptitle(f"Grad-CAM — {row['cell_line']} | {row['culture_medium']} | "
                 f"{row['formation_method']} | {row['timepoint']}", fontsize=12)
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(CFG.output_dir, save_name), dpi=120,
                    bbox_inches="tight")
    plt.show()


# Run for all 10 samples
for i, row in enumerate(samples):
    print(f"\n--- Sample {i+1}: {row['cell_line']} ---")
    gradcam_grid_for_sample(row, save_name=f"gradcam_sample_{i+1}_{row['cell_line']}.png")

## Single Original Spheroid → 8-Attribute Heatmap (Main Fig. 5 Style)

Reproduces the exact figure layout from the paper: one original image on the
left, with 8 individual heatmaps (no overlay, raw CAM) arranged 2×4 on the right.

In [ ]:
def gradcam_raw_grid(row, save_name=None):
    img_pil = load_image_rgb(row["full_path"])
    x       = val_tf(img_pil).unsqueeze(0).to(CFG.device)

    fig = plt.figure(figsize=(14, 7))
    gs  = fig.add_gridspec(2, 5, width_ratios=[1.2, 1, 1, 1, 1])

    # Original image spans both rows, first column
    ax0 = fig.add_subplot(gs[:, 0])
    ax0.imshow(img_pil.resize((CFG.image_size, CFG.image_size)), cmap="gray")
    ax0.set_title("Original\nSpheroid", fontsize=10)
    ax0.axis("off")

    # 8 heatmaps in 2x4 grid (columns 1-4)
    positions = [(r, c) for r in range(2) for c in range(1, 5)]
    for (r, c), label in zip(positions, GRID_ORDER):
        ax  = fig.add_subplot(gs[r, c])
        cam, _ = gradcam(x, label)
        cam_resized = np.array(Image.fromarray((cam*255).astype(np.uint8))
                               .resize((CFG.image_size, CFG.image_size)))
        ax.imshow(cam_resized, cmap="jet")
        ax.set_title(DISPLAY_NAMES.get(label, label), fontsize=10)
        ax.axis("off")

    plt.suptitle(f"Grad-CAM Visualizations — {row['cell_line']}", fontsize=12)
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(CFG.output_dir, save_name), dpi=120,
                    bbox_inches="tight")
    plt.show()


# Reproduce paper-style Fig 5 for the first representative sample
gradcam_raw_grid(samples[0], save_name="figure5_gradcam.png")

## Interpretation Summary

- **Cell line**: attention concentrates on global morphology and boundary structure
- **Formation method / Seeding density**: focus on compactness and internal organisation
- **Timepoint**: later timepoints emphasise dense necrotic cores
- **Biological replicate**: diffuse / background-oriented attention → sensitivity to artefacts
- **Microscope / Magnification**: attention adapts to acquisition-specific cues (global at low res, fine boundary at high res)

These results confirm CoAtNet relies on **interpretable morphological cues** for
biological attributes, while revealing some vulnerability to **technical confounders**
in replicate-level predictions — consistent with Section B of the paper.